# Valeria: fine-tune Gemma 4 E2B (text + photos)

Runtime → Change runtime type → **T4 GPU** (L4/A100 is faster). Run the cells top to bottom.

What it does:
1. builds the text dataset and downloads the CC BY 4.0 wound/sprain photos (or restores both from Drive);
2. scores the **base** model (baseline);
3. trains run 1 (text only) and run 2 (text + photos), scoring each one.

**Everything is written to Google Drive (`MyDrive/valeria_runs`)**, so a disconnect loses at most 50 training steps.
After a disconnect or in a new session, run the cells from the top again: data is restored from Drive, and
training cells resume from their last checkpoint. Skip cells whose results are already in Drive.

In [ ]:
!nvidia-smi

## 1. Get the code and mount Drive
If the GitHub repo is private, either add a GitHub token, or upload a zip of the repo to Drive and unzip it here instead.

In [ ]:
!git clone https://github.com/tommy-Samy123/valeria.git
%cd /content/valeria
!git pull

In [ ]:
!pip install -q -r training/requirements-train.txt -r training/requirements-data.txt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/valeria_runs"  # everything that must survive a disconnect goes here
!mkdir -p {D}

## 2. Data
First session: builds the text dataset, downloads the photos (needs your Roboflow API key as a Colab secret named
`ROBOFLOW_API_KEY`, key icon in the left sidebar; never paste it into a cell) and saves both to `data.zip` in Drive.
Later sessions: restores `data.zip`, so every run is trained and scored on exactly the same examples.

In [ ]:
import os
from pathlib import Path

def sh(cmd):  # like `!cmd`, but stops the cell if the command fails
    ip = get_ipython()
    ip.system(cmd)
    if ip.user_ns.get("_exit_code"):
        raise RuntimeError(f"failed: {cmd}")

if Path(f"{D}/data.zip").exists():
    sh(f"unzip -qo {D}/data.zip -d training/")
    print("restored training/data from Drive")
else:
    from google.colab import userdata
    os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
    sh("python training/scripts/build_text_dataset.py")
    sh("python training/scripts/download_images.py")
    sh("python training/scripts/prepare_images.py --max-per-class 600")
    sh(f"cd training && zip -qr {D}/data.zip data")
    print("saved training/data to Drive")
sh("cat training/data/images/stats.json")

## 3. Smoke test (a few minutes): checks that loading, collation and saving all work

In [ ]:
!python training/scripts/train_gemma4.py --max-steps 10 \
  --train training/data/text/train.jsonl training/data/images/train.jsonl \
  --val training/data/text/val.jsonl --val-limit 20 --out training/outputs/smoke

## 4. Baseline: base model, no fine-tuning

In [ ]:
!python training/scripts/evaluate.py --model unsloth/gemma-4-E2B-it --limit 150 \
  --data training/data/text/test.jsonl training/data/images/test.jsonl --out {D}/eval_base

## 5. Run 1: text only
Re-running this cell after a disconnect continues from the last checkpoint.

In [ ]:
!python training/scripts/train_gemma4.py --resume \
  --train training/data/text/train.jsonl --val training/data/text/val.jsonl --out {D}/text_r16

In [ ]:
!python training/scripts/evaluate.py --model {D}/text_r16/lora \
  --data training/data/text/test.jsonl training/data/images/test.jsonl --out {D}/eval_text_r16

## 6. Run 2: text + photos
Re-running this cell after a disconnect continues from the last checkpoint. The merged 16-bit model (about 10 GB,
needed only for export) is written to the Colab disk, not Drive; copy it out yourself if you have the space.

In [ ]:
!python training/scripts/train_gemma4.py --resume --merge --merge-dir training/outputs/mix_r16_merged \
  --train training/data/text/train.jsonl training/data/images/train.jsonl \
  --val training/data/text/val.jsonl training/data/images/val.jsonl --out {D}/mix_r16

In [ ]:
!python training/scripts/evaluate.py --model {D}/mix_r16/lora \
  --data training/data/text/test.jsonl training/data/images/test.jsonl --out {D}/eval_mix_r16